![KAUST Academy](https://i.imgur.com/a3uAqnb.png)

# Day 16 -- Lab 1: Clustering -- K-Means vs DBSCAN

**Scenario:** A research lab has collected sensor readings from an experiment. The data forms unusual patterns -- some are round blobs, some are crescent-shaped, and some are concentric rings. Your job is to cluster these patterns using two algorithms: **K-Means** and **DBSCAN**. You will discover that shape matters -- the right algorithm depends on the geometry of your data.

| Part | Goal |
|---|---|
| Part 1 | Generate synthetic datasets with different shapes |
| Part 2 | K-Means on blobs -- the easy case |
| Part 3 | The Elbow Method -- choosing the best K |
| Part 4 | K-Means on non-convex shapes -- where it fails |
| Part 5 | DBSCAN -- density-based clustering |
| Part 6 | Silhouette Score -- comparing both algorithms |
| Part 7 | Final showdown on all datasets |

---

## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans, DBSCAN
from sklearn.datasets import make_blobs, make_moons, make_circles
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score

plt.rcParams['figure.dpi'] = 120
np.random.seed(42)

---

## Part 1: Generate Synthetic Datasets

We will create three datasets with very different shapes. This lets us test where each algorithm shines and where it struggles.

In [ ]:
# Dataset 1: Three round blobs (easy for K-Means)
X_blobs, y_blobs = make_blobs(n_samples=300, centers=3,
                               cluster_std=0.8, random_state=42)

# Dataset 2: Two crescent moons (non-convex)
X_moons, y_moons = make_moons(n_samples=300, noise=0.08, random_state=42)

# Dataset 3: Two concentric circles (non-convex)
X_circles, y_circles = make_circles(n_samples=300, noise=0.05,
                                     factor=0.5, random_state=42)

In [ ]:
# Visualize all three datasets
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

datasets = [(X_blobs, y_blobs, "Blobs"),
            (X_moons, y_moons, "Moons"),
            (X_circles, y_circles, "Circles")]

for ax, (X, y, title) in zip(axes, datasets):
    ax.scatter(X[:, 0], X[:, 1], c=y, cmap='viridis', s=20, alpha=0.7)
    ax.set_title(f"{title} (true labels)")
    ax.set_aspect('equal')

plt.tight_layout()
plt.show()

print(f"Blobs:   {X_blobs.shape[0]} points, {len(np.unique(y_blobs))} true clusters")
print(f"Moons:   {X_moons.shape[0]} points, {len(np.unique(y_moons))} true clusters")
print(f"Circles: {X_circles.shape[0]} points, {len(np.unique(y_circles))} true clusters")

---

## Part 2: K-Means on Blobs -- The Easy Case

### Task 1

Apply K-Means with `n_clusters=3` on `X_blobs`. Use `random_state=42`.

1. Fit the model and get the predicted labels using `.fit_predict()`
2. Create a scatter plot colored by the predicted cluster labels
3. Plot the cluster centers as large black "X" markers using `kmeans.cluster_centers_`
4. Title: "K-Means on Blobs (K=3)"

In [ ]:
# Your code here

### Task 2

Print the coordinates of the 3 cluster centers. Then print the **inertia** (`kmeans.inertia_`), which is the total squared distance from each point to its assigned center.

Lower inertia = tighter clusters.

In [ ]:
# Your code here

---

## Part 3: The Elbow Method -- Choosing the Best K

In real data, you don't know how many clusters exist. The **Elbow Method** helps you decide: run K-Means for K = 1, 2, 3, ... and plot the inertia. Look for the "elbow" -- the point where adding more clusters stops helping much.

### Task 3

Run K-Means on `X_blobs` for K = 1 through 10. Store the inertia for each K.

```python
inertias = []
K_range = range(1, 11)

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42)
    km.fit(X_blobs)
    inertias.append(km.inertia_)
```

Plot inertia vs. K as a line plot with circle markers. Label axes and add a title "Elbow Method -- Blobs". At which K do you see the elbow?

In [ ]:
# Your code here

### Task 4

Now do the same elbow analysis on `X_moons`. Run K-Means for K = 1 through 10 and plot the inertia curve.

Is there a clear elbow? Why or why not? (Think about the shape of the moons dataset.)

In [ ]:
# Your code here

---

## Part 4: K-Means on Non-Convex Shapes -- Where It Fails

### Task 5

Apply K-Means with `n_clusters=2` on **both** `X_moons` and `X_circles`. Create a figure with 2 subplots (1 row, 2 columns):
- Left: K-Means on Moons
- Right: K-Means on Circles

Color each point by its predicted label. Plot the centers as black "X" markers. Does K-Means correctly separate the two groups in each case?

In [ ]:
# Your code here

---

## Part 5: DBSCAN -- Density-Based Clustering

DBSCAN does not need K. Instead, it finds clusters by looking for **dense regions** separated by sparse areas. It has two parameters:
- `eps`: the radius of the neighborhood around each point
- `min_samples`: how many neighbors a point needs to be a "core point"

Points that are in sparse regions get labeled as **noise** (label = -1).

### Task 6

Apply DBSCAN to `X_moons` with `eps=0.2` and `min_samples=5`. Get the labels. Then:

1. Print how many clusters were found (hint: `len(set(labels) - {-1})`)
2. Print how many points were labeled as noise (label == -1)
3. Create a scatter plot colored by the predicted labels. Use a different color for noise points (e.g., gray with marker "x")

In [ ]:
# Your code here

### Task 7

Apply DBSCAN to `X_circles` with `eps=0.2` and `min_samples=5`. Create the same kind of plot. Does DBSCAN separate the inner circle from the outer circle?

In [ ]:
# Your code here

### Task 8

DBSCAN is sensitive to its parameters. Try three different `eps` values on `X_moons`: **0.1, 0.2, 0.5**. Keep `min_samples=5`.

Create a figure with 3 subplots (1 row, 3 columns), one for each `eps`. Title each subplot with the eps value and the number of clusters found. What happens when `eps` is too small? Too large?

In [ ]:
# Your code here

---

## Part 6: Silhouette Score -- Comparing Both Algorithms

The **silhouette score** measures how well each point fits in its cluster:
- **+1**: perfectly matched to its own cluster
- **0**: on the boundary between clusters
- **-1**: assigned to the wrong cluster

We compute the average silhouette score across all points. Higher = better clustering.

### Task 9

Compute the silhouette score for K-Means on `X_blobs` for K = 2 through 8. Plot the silhouette score vs. K.

```python
from sklearn.metrics import silhouette_score

sil_scores = []
K_range = range(2, 9)

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42)
    labels = km.fit_predict(X_blobs)
    score = silhouette_score(X_blobs, labels)
    sil_scores.append(score)
```

Which K gives the highest silhouette score? Does it match the elbow from Task 3?

In [ ]:
# Your code here

### Task 10

Now compare K-Means and DBSCAN on all three datasets using the silhouette score. For each dataset:
- K-Means: use the true number of clusters (K=3 for blobs, K=2 for moons, K=2 for circles)
- DBSCAN: use `eps=0.5, min_samples=5` for blobs; `eps=0.2, min_samples=5` for moons and circles

Print a table like this:
```
Dataset   | K-Means Silhouette | DBSCAN Silhouette | Winner
----------|-------------------|-------------------|-------
Blobs     |       ???         |       ???         |  ???
Moons     |       ???         |       ???         |  ???
Circles   |       ???         |       ???         |  ???
```

**Important:** DBSCAN may label some points as noise (-1). When computing the silhouette score for DBSCAN, only include non-noise points:
```python
mask = db_labels != -1
if len(set(db_labels[mask])) > 1:
    score = silhouette_score(X[mask], db_labels[mask])
```

In [ ]:
# Your code here

---

## Part 7: Final Showdown -- Side-by-Side Comparison

Let's put it all together with one big comparison figure.

### Task 11

Create a figure with **3 rows and 3 columns** (`figsize=(16, 14)`):
- **Row 1:** True labels for each dataset
- **Row 2:** K-Means results for each dataset
- **Row 3:** DBSCAN results for each dataset

Columns: Blobs, Moons, Circles.

Use the same K-Means and DBSCAN parameters from Task 10. Add the silhouette score to each subplot title (e.g., "K-Means on Moons (sil=0.38)"). For DBSCAN, also show the number of noise points.

Use `ax.set_aspect('equal')` so the shapes are not distorted.

In [ ]:
# Your code here

---

## Wrap-Up

**Key takeaways:**

- **K-Means** assigns every point to the nearest of K centers. It works well on round, well-separated blobs but fails on non-convex shapes.
- The **Elbow Method** (plot inertia vs. K) and the **Silhouette Score** (higher is better) help you choose the right K.
- **DBSCAN** finds clusters by density. It handles arbitrary shapes and automatically detects noise points, but you need to tune `eps` and `min_samples`.
- Always visualize your data and try both algorithms -- no single method works best on every dataset.
- **Next:** Hierarchical extend these ideas further.